In [14]:
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    Settings,
    get_response_synthesizer,
)
from llama_parse import LlamaParse
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.ollama import Ollama
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.evaluation import (
    RelevancyEvaluator,
    CorrectnessEvaluator,
)
import pandas as pd
import asyncio
import json
import time
import nest_asyncio
from types import SimpleNamespace
from llama_index.core.llama_dataset import (
    LabelledRagDataExample,
)
import itertools

# Apply nested asyncio for notebook execution
nest_asyncio.apply()


class RAGEvaluator:
    def __init__(self, data_directory="data", dataset_file="./rag_dataset.json"):
        # Configure settings
        self._configure_settings()

        # Initialize parser and file extractor
        self.parser = LlamaParse(result_type="markdown")
        self.file_extractor = {".pdf": self.parser}

        # Load documents
        self.documents = self._load_documents(data_directory)

        # Load dataset
        self.rag_dataset = self._load_dataset(dataset_file)

        # Initialize evaluators
        self.relevancy_evaluator = RelevancyEvaluator()
        self.correctness_evaluator = CorrectnessEvaluator()

        # Initialize results DataFrame
        self.eval_results_df = pd.DataFrame()

        # Initialize summary results DataFrame for different parameter combinations
        self.summary_results_df = pd.DataFrame()

    def _configure_settings(self):
        """Configure global settings for embedding model and LLM"""
        Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-large-zh-v1.5")
        Settings.llm = Ollama(
            model="llama3.1:latest", request_timeout=60.0, temperature=0.3
        )

    def _load_documents(self, data_directory):
        """Load documents from the specified directory"""
        return SimpleDirectoryReader(
            data_directory, file_extractor=self.file_extractor
        ).load_data()

    def _load_dataset(self, dataset_file):
        """Load the RAG dataset from a JSON file"""
        with open(dataset_file, "r") as f:
            # Convert dictionary to object with attribute access
            return json.load(f, object_hook=lambda d: SimpleNamespace(**d))

    def _create_query_engine(self, chunk_size, chunk_overlap, top_k):
        """Create a query engine with the specified parameters"""
        # Create text splitter with the specified chunk size and overlap
        text_splitter = SentenceSplitter(
            chunk_size=chunk_size, chunk_overlap=chunk_overlap
        )

        # Create vector index
        index = VectorStoreIndex.from_documents(
            self.documents, transformations=[text_splitter]
        )

        # Configure retrievers with the specified top_k
        vector_retriever = index.as_retriever(similarity_top_k=top_k, verbose=True)
        bm25_retriever = BM25Retriever.from_defaults(
            docstore=index.docstore, similarity_top_k=top_k
        )

        # Create fusion retriever
        retriever = QueryFusionRetriever(
            [vector_retriever, bm25_retriever],
            similarity_top_k=top_k,
            num_queries=1,  # set to 1 to disable query generation
            mode="reciprocal_rerank",
            use_async=True,
            verbose=True,
        )

        # Create response synthesizer
        response_synthesizer = get_response_synthesizer()

        # Create and return query engine
        return RetrieverQueryEngine(
            retriever=retriever,
            response_synthesizer=response_synthesizer,
        )

    def _add_eval_row(
        self,
        response,
        dataset_example,
        relevancy_result,
        correctness_result,
        response_time,
    ):
        """Add evaluation results to the DataFrame"""
        if not response.source_nodes:
            print("No response!")
            return

        eval_row = pd.DataFrame(
            [
                {
                    "Query": dataset_example.query,
                    "Response": str(response),
                    "Reference Answer": dataset_example.reference_answer,
                    "Source": response.source_nodes[0].node.text[:1000] + "...",
                    "Relevancy Eval Result": f"{'Pass' if relevancy_result.passing else 'Fail'} \nscore: {relevancy_result.score}",
                    "Relevancy Reasoning": relevancy_result.feedback,
                    "Correctness Eval Result": f"{'Pass' if correctness_result.passing else 'Fail'} \n\nscore: {correctness_result.score}",
                    "Correctness Reasoning": correctness_result.feedback,
                    "Response Time": f"{response_time:.4f}",
                }
            ]
        )

        self.eval_results_df = pd.concat(
            [self.eval_results_df, eval_row], ignore_index=True
        )

    async def _evaluate_engine(
        self, query_engine, dataset_examples: LabelledRagDataExample
    ):
        """Evaluate the query engine on the given examples"""
        # Limit to first 3 examples for testing
        dataset_examples = dataset_examples[:3]

        # Initialize tracking variables
        relevancy_total_correct = 0
        correctness_total_correct = 0
        correctness_total_score = 0
        total_response_time = 0

        # Process each example
        for dataset_example in dataset_examples:
            start_time = time.time()
            response = query_engine.query(dataset_example.query)
            response_time = time.time() - start_time

            # Evaluate relevancy and correctness
            relevancy_result = self.relevancy_evaluator.evaluate_response(
                query=dataset_example.query, response=response
            )
            correctness_result = self.correctness_evaluator.evaluate_response(
                query=dataset_example.query,
                response=response,
                reference=dataset_example.reference_answer,
            )

            # Add results to DataFrame
            self._add_eval_row(
                response,
                dataset_example,
                relevancy_result,
                correctness_result,
                response_time,
            )

            # Update totals
            total_response_time += response_time
            if relevancy_result.passing:
                relevancy_total_correct += 1
            if correctness_result.passing:
                correctness_total_correct += 1
                correctness_total_score += correctness_result.score

        return (
            relevancy_total_correct,
            correctness_total_correct,
            correctness_total_score,
            len(dataset_examples),
            total_response_time,
        )

    def evaluate_with_params(self, chunk_size, chunk_overlap, top_k):
        """Run evaluation with the specified parameters"""
        print(
            f"Parameters: chunk_size={chunk_size}, chunk_overlap={chunk_overlap}, top_k={top_k}"
        )

        # Reset results DataFrame
        self.eval_results_df = pd.DataFrame()

        # Create query engine with the specified parameters
        query_engine = self._create_query_engine(chunk_size, chunk_overlap, top_k)

        # Run evaluation
        (
            relevancy_total_correct,
            correctness_total_correct,
            correctness_total_score,
            total_questions,
            total_response_time,
        ) = asyncio.run(self._evaluate_engine(query_engine, self.rag_dataset.examples))

        # Display results
        styled_df = self.eval_results_df.style.set_properties(
            **{"white-space": "pre-wrap"},
        )

        display(styled_df)

        # Calculate scores
        relevancy_score = relevancy_total_correct / total_questions
        correctness_score = correctness_total_score / total_questions
        avg_response_time = total_response_time / total_questions

        # Display summary
        print(
            f"Total Relevancy correct: {relevancy_total_correct} out of {total_questions}, "
            f"score: {relevancy_score}"
        )
        print(
            f"Total Correctness correct: {correctness_total_correct} out of {total_questions}, "
            f"score: {correctness_score}"
        )
        print(f"Average response time: {avg_response_time:.4f} seconds")
        print("===============================================")

        # Add result to summary DataFrame
        self._add_summary_row(
            chunk_size,
            chunk_overlap,
            top_k,
            relevancy_score,
            correctness_score,
            avg_response_time,
        )

        return {
            "relevancy_score": relevancy_score,
            "correctness_score": correctness_score,
            "avg_response_time": avg_response_time,
        }

    def _add_summary_row(
        self,
        chunk_size,
        chunk_overlap,
        top_k,
        relevancy_score,
        correctness_score,
        avg_response_time,
    ):
        """Add a summary row to the summary results DataFrame"""
        summary_row = pd.DataFrame(
            [
                {
                    "Chunk Size": chunk_size,
                    "Chunk Overlap": chunk_overlap,
                    "Top K": top_k,
                    "Relevancy Score": f"{relevancy_score:.4f}",
                    "Correctness Score": f"{correctness_score:.4f}",
                    "Avg Response Time": f"{avg_response_time:.4f}",
                }
            ]
        )

        self.summary_results_df = pd.concat(
            [self.summary_results_df, summary_row], ignore_index=True
        )

    def run_evaluations(self, chunk_sizes, overlaps, top_ks):
        """Run evaluations for multiple parameter combinations"""
        # Reset summary results DataFrame
        self.summary_results_df = pd.DataFrame()

        # Generate all parameter combinations
        param_combinations = list(itertools.product(chunk_sizes, overlaps, top_ks))
        total_combinations = len(param_combinations)

        print(f"Running evaluations for {total_combinations} parameter combinations...")

        results = {}
        for i, (chunk_size, overlap, top_k) in enumerate(param_combinations):
            print(f"\nEvaluation {i+1}/{total_combinations}")
            # Convert overlap from percentage to absolute value
            chunk_overlap = int(chunk_size * overlap)
            results[(chunk_size, overlap, top_k)] = self.evaluate_with_params(
                chunk_size, chunk_overlap, top_k
            )

        # Display summary table
        self._display_summary_table()

        return results

    def _display_summary_table(self):
        """Display a summary table of all parameter combinations"""
        print("\n--- Summary of All Parameter Combinations ---")

        # Sort the summary results by scores (you can change the sorting criteria)
        sorted_df = self.summary_results_df.sort_values(
            by=["Relevancy Score", "Correctness Score"], ascending=False
        )

        display(sorted_df)

        # Find the best parameter combination
        best_row = sorted_df.iloc[0]
        print(f"\nBest Parameter Combination:")
        print(f"Chunk Size: {best_row['Chunk Size']}")
        print(f"Chunk Overlap: {best_row['Chunk Overlap']}")
        print(f"Top K: {best_row['Top K']}")
        print(f"Relevancy Score: {best_row['Relevancy Score']}")
        print(f"Correctness Score: {best_row['Correctness Score']}")
        print(f"Avg Response Time: {best_row['Avg Response Time']} seconds")


# Usage example
if __name__ == "__main__":
    evaluator = RAGEvaluator()

    # Define parameter ranges to test
    chunk_sizes = [1024,2048]
    overlaps = [0.1, 0.15]  # 10% and 15% overlap
    top_ks = [3, 4]

    results = evaluator.run_evaluations(chunk_sizes, overlaps, top_ks)

Started parsing the file under job_id 88b5c9a0-bda4-480a-ae9d-f5fe1bd74266
Started parsing the file under job_id c8b10dc5-d288-4991-bb87-05bda88f150a
Started parsing the file under job_id 32cdcc85-6186-4833-840c-17264847098b
Started parsing the file under job_id 8f1677cf-85f9-4106-b33c-4832680de9b6
Started parsing the file under job_id c05b3586-07a9-477f-9fa6-0e4e8ff51192
Started parsing the file under job_id ccd633bf-ba53-440c-a51c-ca0715cb6a49
Running evaluations for 8 parameter combinations...

Evaluation 1/8
Parameters: chunk_size=1024, chunk_overlap=102, top_k=3


,Query,Response,Reference Answer,Source,Relevancy Eval Result,Relevancy Reasoning,Correctness Eval Result,Correctness Reasoning,Response Time
0,什麼是學餐資訊？,這是一份關於學校食堂的資訊，內容包括了各種不同主題的餐廳和招牌菜式。,學餐資訊是指學校內的學生餐廳提供的餐點資訊，包括不同菜式、營養特點等。,# 學餐資訊 為倡導SDGs的概念，校內的學生餐廳也是採用其中幾個目標的概念來建立： # 22 ZERO HUNGER # 1. 每日學餐（強調SDGs目標1：零飢餓 Zero Hunger） - (1) 每日惜食餐廳 - 提供當日生鮮剩餘食材製作的健康餐點，減少食物浪費。 - 招牌菜：惜食蔬菜濃湯、健康全麥三明治。 - (2) 有機農場餐廳 - 使用校園農業計劃種植的有機蔬菜，提供新鮮、健康的餐點。 - 招牌菜：有機沙拉碗、田園蔬菜披薩。 - (3) 健康小食堂 - 專為有需要的學生提供免費或低價的營養餐點。 - 招牌菜：營養便當、五穀雜糧飯。 # 12 RESPONSIBLE CONSUMPTION AND PRODUCTION # 2. Eco Eats學餐（強調SDGs目標12：負責任的消費和生產） - (1) 綠色餐廳 - 提供本地採購的食材，減少食物碳足跡。 - 招牌菜：本地風味烤雞、鮮果沙拉。 - (2) 廢物再利用餐廳 - 使用食材的每一部分，推動零廢棄理念。 - 招牌菜：蔬菜莖葉炒飯、柑橘皮蜜餞。 - (3) 環保快餐店 - 所有餐具均為可再生或可降解材料，提倡環保用餐。 - 招牌菜：竹纖維碗拉麵、紙漿盒三明治。 # 3 GOOD HEALTH AND WELL-BEING # 3. 健康學餐（強調SDGs目標3：良好健康與福祉） - (1) 運動員餐廳 - 提供高蛋白、低脂肪的餐點，適合運動後補充能量。 - 招牌菜：健身沙拉、雞胸肉餐盒。 - (2) 心靈健康餐廳 - 提供低糖、低鹽的餐點，支持心理健康。 - 招牌菜：穀物水果碗、香菇豆腐湯。 - (3) 健康果汁吧 - 提供新鮮水果和蔬菜汁，增強免疫力。 - 招牌菜：綠色活力汁、莓果奶昔。...,Pass score: 1.0,YES.,Fail score: 2.0,"The generated answer is relevant to the user query, as it mentions ""學校食堂"", which is related to ""學餐資訊"". However, the content of the generated answer is not accurate and does not provide specific information about the menu or nutritional features of school meals, unlike the reference answer.",5.2948
1,校內的學生餐廳採用哪些SDGs目標的概念來建立？,校內的學生餐廳採用了SDGs目標1：零飢餓、目標12：負責任的消費和生產，以及目標3：良好健康與福祉的概念來建立。,根據提供的文本，校內的學生餐廳採用了以下幾個SDGs目標的概念來建立： 1. SDGs目標1：零飢餓 (Zero Hunger) 2. SDGs目標12：負責任的消費和生產 3. SDGs目標3：良好健康與福祉,"# SUSTAINABLE CITIES AND COMMUNITIES # 4. Green Garden學餐（強調SDGs目標11：可持續城市和社區） # (1) 社區農園餐廳 - 利用校園社區農園的蔬菜，提供新鮮健康的餐點。 - 招牌菜：社區農園沙拉、蔬菜焗飯。 # (2) 綠建築餐廳 - 餐廳建築採用可再生能源，並設有綠色屋頂。 - 招牌菜：太陽能烤肉、風能披薩。 # CLEAN WATER AND SANITATION # 5. Clean Water Café （強調SDGs目標6：清潔飲水和衛生設施） # (1) 純淨水吧 - 提供純淨飲用水和各類健康飲品，所有水源經過嚴格檢測。 - 招牌菜：純淨水冷泡茶、健康果蔬水。 # (2) 衛生餐廳 - 推動個人衛生教育，所有餐點均在高度衛生環境下製作。 - 招牌菜：無菌沙拉、潔淨便當。 # PEACE, JUSTICE AND STRONG INSTITUTIONS # 6. Peaceful Plates（強調SDGs目標16：和平、正義和強大機構） # (1) 公平貿易餐廳 - 所有食材均來自公平貿易供應商，支持全球正義。 - 招牌菜：公平貿易咖啡、道德巧克力蛋糕。 # (2) 和平餐廳 - 提供來自各地區的和平美食，推動文化交流與理解。 - 招牌菜：和平三明治、全球風味拼盤。 # (3) 社會企業餐廳 - 與社會企業合作，支持社會弱勢群體。 - 招牌菜：慈善義大利麵、公益蔬菜湯。...",Pass score: 1.0,"YES. The response mentions that the school's student cafeteria adopts SDGs targets 1, 12, and 3 for its establishment, which is in line with the context information provided about the school's efforts to promote sustainable development goals (SDGs) through various activities and initiatives, including the use of SDG-related concepts in their cafeteria.",Pass score: 4.0,"The generated answer is fully relevant to the user query and accurately lists the SDGs targets adopted by the student cafeteria, matching the reference answer in terms of content. The only difference is that the reference answer provides a brief description for each target, but the generated answer still conveys the same information without any mistakes.",1.7272
2,什麼是 Green Garden 學餐？,社區農園餐廳。,社區農園餐廳。,"# SUSTAINABLE CITIES AND COMMUNITIES # 4. Green Garden學餐（強調SDGs目標11：可持續城市和社區） # (1) 社區農園餐廳 - 利用校園社區農園的蔬菜，提供新鮮健康的餐點。 - 招牌菜：社區農園沙拉、蔬菜焗飯。 # (2) 綠建築餐廳 - 餐廳建築採用可再生能源，並設有綠色屋頂。 - 招牌菜：太陽能烤肉、風能披薩。 # CLEAN WATER AND SANITATION # 5. Clean Water Café （強調SDGs目標6：清潔飲水和衛生設施） # (1) 純淨水吧 - 提供純淨飲用水和各類健康飲品，所有水源經過嚴格檢測。 - 招牌菜：純淨水冷泡茶、健康果蔬水。 # (2) 衛生餐廳 - 推動個人衛生教育，所有餐點均在高度衛生環境下製作。 - 招牌菜：無菌沙拉、潔淨便當。 # PEACE, JUSTICE AND STRONG INSTITUTIONS # 6. Peaceful Plates（強調SDGs目標16：和平、正義和強大機構） # (1) 公平貿易餐廳 - 所有食材均來自公平貿易供應商，支持全球正義。 - 招牌菜：公平貿易咖啡、道德巧克力蛋糕。 # (2) 和平餐廳 - 提供來自各地區的和平美食，推動文化交流與理解。 - 招牌菜：和平三明治、全球風味拼盤。 # (3) 社會企業餐廳 - 與社會企業合作，支持社會弱勢群體。 - 招牌菜：慈善義大利麵、公益蔬菜湯。...",Pass score: 1.0,YES.,Pass score: 5.0,"The generated answer is identical to the reference answer, indicating that it is fully correct and relevant to the user query. The fact that both answers are concise also supports a high score.",1.0173


Total Relevancy correct: 3 out of 3, score: 1.0
Total Correctness correct: 2 out of 3, score: 3.0
Average response time: 2.6798 seconds

Evaluation 2/8
Parameters: chunk_size=1024, chunk_overlap=102, top_k=4


,Query,Response,Reference Answer,Source,Relevancy Eval Result,Relevancy Reasoning,Correctness Eval Result,Correctness Reasoning,Response Time
0,什麼是學餐資訊？,這是一份關於學校食堂的資訊手冊，內容包括了各種不同類型的餐廳和菜單。,學餐資訊是指學校內的學生餐廳提供的餐點資訊，包括不同菜式、營養特點等。,# 學餐資訊 為倡導SDGs的概念，校內的學生餐廳也是採用其中幾個目標的概念來建立： # 22 ZERO HUNGER # 1. 每日學餐（強調SDGs目標1：零飢餓 Zero Hunger） - (1) 每日惜食餐廳 - 提供當日生鮮剩餘食材製作的健康餐點，減少食物浪費。 - 招牌菜：惜食蔬菜濃湯、健康全麥三明治。 - (2) 有機農場餐廳 - 使用校園農業計劃種植的有機蔬菜，提供新鮮、健康的餐點。 - 招牌菜：有機沙拉碗、田園蔬菜披薩。 - (3) 健康小食堂 - 專為有需要的學生提供免費或低價的營養餐點。 - 招牌菜：營養便當、五穀雜糧飯。 # 12 RESPONSIBLE CONSUMPTION AND PRODUCTION # 2. Eco Eats學餐（強調SDGs目標12：負責任的消費和生產） - (1) 綠色餐廳 - 提供本地採購的食材，減少食物碳足跡。 - 招牌菜：本地風味烤雞、鮮果沙拉。 - (2) 廢物再利用餐廳 - 使用食材的每一部分，推動零廢棄理念。 - 招牌菜：蔬菜莖葉炒飯、柑橘皮蜜餞。 - (3) 環保快餐店 - 所有餐具均為可再生或可降解材料，提倡環保用餐。 - 招牌菜：竹纖維碗拉麵、紙漿盒三明治。 # 3 GOOD HEALTH AND WELL-BEING # 3. 健康學餐（強調SDGs目標3：良好健康與福祉） - (1) 運動員餐廳 - 提供高蛋白、低脂肪的餐點，適合運動後補充能量。 - 招牌菜：健身沙拉、雞胸肉餐盒。 - (2) 心靈健康餐廳 - 提供低糖、低鹽的餐點，支持心理健康。 - 招牌菜：穀物水果碗、香菇豆腐湯。 - (3) 健康果汁吧 - 提供新鮮水果和蔬菜汁，增強免疫力。 - 招牌菜：綠色活力汁、莓果奶昔。...,Pass score: 1.0,YES.,Pass score: 4.0,"The generated answer is relevant to the user query, as it mentions ""學餐資訊"" and provides information about school cafeterias. However, it contains some inaccuracies, such as mentioning a ""資訊手冊"" (information manual) instead of specific menu information, which is what the reference answer describes. The score reflects that the generated answer is mostly relevant but has some mistakes.",3.6251
1,校內的學生餐廳採用哪些SDGs目標的概念來建立？,校內的學生餐廳採用了零飢餓（Zero Hunger）、負責任的消費和生產（Responsible Consumption and Production），以及良好健康與福祉（Good Health and Well-being）等SDGs目標的概念來建立。,根據提供的文本，校內的學生餐廳採用了以下幾個SDGs目標的概念來建立： 1. SDGs目標1：零飢餓 (Zero Hunger) 2. SDGs目標12：負責任的消費和生產 3. SDGs目標3：良好健康與福祉,"# SUSTAINABLE CITIES AND COMMUNITIES # 4. Green Garden學餐（強調SDGs目標11：可持續城市和社區） # (1) 社區農園餐廳 - 利用校園社區農園的蔬菜，提供新鮮健康的餐點。 - 招牌菜：社區農園沙拉、蔬菜焗飯。 # (2) 綠建築餐廳 - 餐廳建築採用可再生能源，並設有綠色屋頂。 - 招牌菜：太陽能烤肉、風能披薩。 # CLEAN WATER AND SANITATION # 5. Clean Water Café （強調SDGs目標6：清潔飲水和衛生設施） # (1) 純淨水吧 - 提供純淨飲用水和各類健康飲品，所有水源經過嚴格檢測。 - 招牌菜：純淨水冷泡茶、健康果蔬水。 # (2) 衛生餐廳 - 推動個人衛生教育，所有餐點均在高度衛生環境下製作。 - 招牌菜：無菌沙拉、潔淨便當。 # PEACE, JUSTICE AND STRONG INSTITUTIONS # 6. Peaceful Plates（強調SDGs目標16：和平、正義和強大機構） # (1) 公平貿易餐廳 - 所有食材均來自公平貿易供應商，支持全球正義。 - 招牌菜：公平貿易咖啡、道德巧克力蛋糕。 # (2) 和平餐廳 - 提供來自各地區的和平美食，推動文化交流與理解。 - 招牌菜：和平三明治、全球風味拼盤。 # (3) 社會企業餐廳 - 與社會企業合作，支持社會弱勢群體。 - 招牌菜：慈善義大利麵、公益蔬菜湯。...",Pass score: 1.0,"YES. The response mentions that the school's student cafeteria adopts concepts from SDGs targets such as Zero Hunger, Responsible Consumption and Production, and Good Health and Well-being. This is in line with the context information provided, which describes various initiatives and activities on campus related to sustainability and social responsibility, including the adoption of SDG-related concepts in different aspects of campus life.",Pass score: 4.0,"The generated answer is fully relevant to the user query, as it mentions three specific SDGs targets that are adopted by the student cafeteria. The answer also has the same level of detail as the reference answer, with no major mistakes or omissions. However, I deduct a small point because the generated answer uses slightly more concise language than the reference answer, which is not necessarily a bad thing. Overall, the generated answer is very accurate and informative, making it deserving of a high score.",1.7773
2,什麼是 Green Garden 學餐？,Green Garden 學餐是一個強調可持續城市和社區的概念，提供新鮮健康的餐點。它利用校園社區農園的蔬菜，並設有綠色屋頂的餐廳建築。招牌菜包括社區農園沙拉、蔬菜焗飯等。,社區農園餐廳。,"# SUSTAINABLE CITIES AND COMMUNITIES # 4. Green Garden學餐（強調SDGs目標11：可持續城市和社區） # (1) 社區農園餐廳 - 利用校園社區農園的蔬菜，提供新鮮健康的餐點。 - 招牌菜：社區農園沙拉、蔬菜焗飯。 # (2) 綠建築餐廳 - 餐廳建築採用可再生能源，並設有綠色屋頂。 - 招牌菜：太陽能烤肉、風能披薩。 # CLEAN WATER AND SANITATION # 5. Clean Water Café （強調SDGs目標6：清潔飲水和衛生設施） # (1) 純淨水吧 - 提供純淨飲用水和各類健康飲品，所有水源經過嚴格檢測。 - 招牌菜：純淨水冷泡茶、健康果蔬水。 # (2) 衛生餐廳 - 推動個人衛生教育，所有餐點均在高度衛生環境下製作。 - 招牌菜：無菌沙拉、潔淨便當。 # PEACE, JUSTICE AND STRONG INSTITUTIONS # 6. Peaceful Plates（強調SDGs目標16：和平、正義和強大機構） # (1) 公平貿易餐廳 - 所有食材均來自公平貿易供應商，支持全球正義。 - 招牌菜：公平貿易咖啡、道德巧克力蛋糕。 # (2) 和平餐廳 - 提供來自各地區的和平美食，推動文化交流與理解。 - 招牌菜：和平三明治、全球風味拼盤。

Total Relevancy correct: 3 out of 3, score: 1.0
Total Correctness correct: 3 out of 3, score: 4.0
Average response time: 2.4833 seconds

Evaluation 3/8
Parameters: chunk_size=1024, chunk_overlap=153, top_k=3


...

Total Relevancy correct: 3 out of 3, score: 1.0
Total Correctness correct: 3 out of 3, score: 4.333333333333333
Average response time: 2.0622 seconds

Evaluation 8/8
Parameters: chunk_size=2048, chunk_overlap=307, top_k=4


,Query,Response,Reference Answer,Source,Relevancy Eval Result,Relevancy Reasoning,Correctness Eval Result,Correctness Reasoning,Response Time
0,什麼是學餐資訊？,為倡導SDGs的概念，校內的學生餐廳也是採用其中幾個目標的概念來建立。,學餐資訊是指學校內的學生餐廳提供的餐點資訊，包括不同菜式、營養特點等。,"# SUSTAINABLE CITIES AND COMMUNITIES # 4. Green Garden學餐（強調SDGs目標11：可持續城市和社區） # (1) 社區農園餐廳 - 利用校園社區農園的蔬菜，提供新鮮健康的餐點。 - 招牌菜：社區農園沙拉、蔬菜焗飯。 # (2) 綠建築餐廳 - 餐廳建築採用可再生能源，並設有綠色屋頂。 - 招牌菜：太陽能烤肉、風能披薩。 # CLEAN WATER AND SANITATION # 5. Clean Water Café （強調SDGs目標6：清潔飲水和衛生設施） # (1) 純淨水吧 - 提供純淨飲用水和各類健康飲品，所有水源經過嚴格檢測。 - 招牌菜：純淨水冷泡茶、健康果蔬水。 # (2) 衛生餐廳 - 推動個人衛生教育，所有餐點均在高度衛生環境下製作。 - 招牌菜：無菌沙拉、潔淨便當。 # PEACE, JUSTICE AND STRONG INSTITUTIONS # 6. Peaceful Plates（強調SDGs目標16：和平、正義和強大機構） # (1) 公平貿易餐廳 - 所有食材均來自公平貿易供應商，支持全球正義。 - 招牌菜：公平貿易咖啡、道德巧克力蛋糕。 # (2) 和平餐廳 - 提供來自各地區的和平美食，推動文化交流與理解。 - 招牌菜：和平三明治、全球風味拼盤。 # (3) 社會企業餐廳 - 與社會企業合作，支持社會弱勢群體。 - 招牌菜：慈善義大利麵、公益蔬菜湯。...",Pass score: 1.0,YES.,Fail score: 2.0,"The generated answer is relevant to the user query, as it mentions the concept of SDGs and its application in the school's student cafeteria. However, the answer contains mistakes, as it does not provide any information about the types of food or nutritional features that are typically associated with ""學餐資訊"" (school meal information). The reference answer provides a clear and accurate description of what ""學餐資訊"" entails, which is not present in the generated answer.",4.0340
1,校內的學生餐廳採用哪些SDGs目標的概念來建立？,校內的學生餐廳採用了SDGs目標1：零飢餓、目標12：負責任的消費和生產、目標3：良好健康與福祉等幾個目標的概念來建立。,根據提供的文本，校內的學生餐廳採用了以下幾個SDGs目標的概念來建立： 1. SDGs目標1：零飢餓 (Zero Hunger) 2. SDGs目標12：負責任的消費和生產 3. SDGs目標3：良好健康與福祉,"# SUSTAINABLE CITIES AND COMMUNITIES # 4. Green Garden學餐（強調SDGs目標11：可持續城市和社區） # (1) 社區農園餐廳 - 利用校園社區農園的蔬菜，提供新鮮健康的餐點。 - 招牌菜：社區農園沙拉、蔬菜焗飯。 # (2) 綠建築餐廳 - 餐廳建築採用可再生能源，並設有綠色屋頂。 - 招牌菜：太陽能烤肉、風能披薩。 # CLEAN WATER AND SANITATION # 5. Clean Water Café （強調SDGs目標6：清潔飲水和衛生設施） # (1) 純淨水吧 - 提供純淨飲用水和各類健康飲品，所有水源經過嚴格檢測。 - 招牌菜：純淨水冷泡茶、健康果蔬水。 # (2) 衛生餐廳 - 推動個人衛生教育，所有餐點均在高度衛生環境下製作。 - 招牌菜：無菌沙拉、潔淨便當。 # PEACE, JUSTICE AND STRONG INSTITUTIONS # 6. Peaceful Plates（強調SDGs目標16：和平、正義和強大機構） # (1) 公平貿易餐廳 - 所有食材均來自公平貿易供應商，支持全球正義。 - 招牌菜：公平貿易咖啡、道德巧克力蛋糕。 # (2) 和平餐廳 - 提供來自各地區的和平美食，推動文化交流與理解。 - 招牌菜：和平三明治、全球風味拼盤。 # (3) 社會企業餐廳 - 與社會企業合作，支持社會弱勢群體。 - 招牌菜：慈善義大利麵、公益蔬菜湯。...",Pass score: 1.0,YES.,Pass score: 4.0,"The generated answer is fully relevant to the user query and contains all the correct information as the reference answer, with a slight difference in formatting but not affecting the correctness of the content. The score reflects the high accuracy and completeness of the generated answer.",2.0935
2,什麼是 Green Garden 學餐？,**Rewrite** Green Garden 學餐是一個推動健康飲食和減少碳足跡的計劃，它利用校園社區農園的生產物，提供新鮮、健康的餐點。這樣做不僅能夠讓學生享受到營養豐富的食物，也能夠教導他們關心環境和節省資源的重要性。,社區農園餐廳。,"# SUSTAINABLE CITIES AND COMMUNITIES # 4. Green Garden學餐（強調SDGs目標11：可持續城市和社區） # (1) 社區農園餐廳 - 利用校園社區農園的蔬菜，提供新鮮健康的餐點。 - 招牌菜：社區農園沙拉、蔬菜焗飯。 # (2) 綠建築餐廳 - 餐廳建築採用可再生能源，並設有綠色屋頂。 - 招牌菜：太陽能烤肉、風能披薩。 # CLEAN WATER AND SANITATION # 5. Clean Water Café （強調SDGs目標6：清潔飲水和衛生設施） # (1) 純淨水吧 - 提供純淨飲用水和各類健康飲品，所有水源經過嚴格檢測。 - 招牌菜：純淨水冷泡茶、健康果蔬水。 # (2) 衛生餐廳 - 推動個人衛生教育，所有餐點均在高度衛生環境下製作。 - 招牌菜：無菌沙拉、潔淨便當。 # PEACE, JUSTICE AND STRONG INSTITUTIONS # 6. Peaceful Plates（強調SDGs目標16：和平、正義和強大機構） # (1) 公平貿易餐廳 - 所有食材均來自公平貿易供應商，支持全球正義。 - 招牌菜：公平貿易咖啡、道德巧克力蛋糕。 # (2) 和平餐廳 - 提供來自各地區的和平美食，推動文化交流與理解。 - 招牌菜：和平三明治、全球風味拼盤。 # (3) 社會企業餐廳 - 與社會企業合作，支持社會弱勢群體。 - 招牌菜：慈善義大利麵、公益蔬菜湯。...",Pass score: 1.0,"YES. The query asks about Green Garden 學餐, and the response mentions that it's a program promoting healthy eating and reducing carbon footprint by using school garden produce to provide fresh and nutritious meals. This aligns with the overall theme of sustainability and environmental awareness in the provided context, which includes initiatives like reducing food waste (目標 2:零廢不留), promoting eco-friendly transportation (目標 3:良好健康與福祉), and encouraging sustainable practices (目標 4:優質教育).",Pass score: 4.0,"The generated answer is highly relevant to the user query, and it provides a detailed explanation of what Green Garden 學餐 is. The answer also m

Total Relevancy correct: 3 out of 3, score: 1.0
Total Correctness correct: 2 out of 3, score: 2.6666666666666665
Average response time: 3.0844 seconds

--- Summary of All Parameter Combinations ---


,Chunk Size,Chunk Overlap,Top K,Relevancy Score,Correctness Score,Avg Response Time
6,2048,307,3,1.0000,4.3333,2.0622
1,1024,102,4,1.0000,4.0000,2.4833
2,1024,153,3,1.0000,4.0000,2.1806
3,1024,153,4,1.0000,4.0000,2.5728
4,2048,204,3,1.0000,4.0000,2.2938
0,1024,102,3,1.0000,3.0000,2.6798
5,2048,204,4,1.0000,2.6667,3.0870
7,2048,307,4,1.0000,2.6667,3.0844



Best Parameter Combination:
Chunk Size: 2048
Chunk Overlap: 307
Top K: 3
Relevancy Score: 1.0000
Correctness Score: 4.3333
Avg Response Time: 2.0622 seconds
